# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [2]:
# import the DFS
import pandas as pd
import numpy as np
df1 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv")
df2 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv")
df3 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv")


In [4]:
print("df1 columns:", df1.columns.tolist())
print("df2 columns:", df2.columns.tolist())
print("df3 columns:", df3.columns.tolist())

df1 columns: ['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Vehicle Class', 'Total Claim Amount']
df2 columns: ['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Total Claim Amount', 'Policy Type', 'Vehicle Class']
df3 columns: ['Customer', 'State', 'Customer Lifetime Value', 'Education', 'Gender', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Total Claim Amount', 'Vehicle Class']


In [7]:
#standarize column names
df1.columns = df1.columns.str.lower()
df2.columns = df2.columns.str.lower()
df3.columns = df3.columns.str.lower()

# Replace abbreviations for consistency
df1 = df1.rename(columns={'st': 'state', 'gender': 'gender'})
df2 = df2.rename(columns={'st': 'state', 'gender': 'gender'})
df3 = df3.rename(columns={'customer lifetime value': 'customer lifetime value'})


In [15]:
# define cleaning function: 
def clean_data(df):
    # 1. Clean customer_lifetime_valus
    if 'customer_lifetime_value' in df.columns:
        df['customer_lifetime_value'] = (
            df['customer_lifetime_value']
            .astype(str)
            .str.replace('%', '', regex=False)
            .astype(float)
        )
    
    # 2. Clean number_of_open_complaints, keep the second value of the expression
    if 'number_of_open_complaints' in df.columns:
        df['number_of_open_complaints'] = (
            df['number_of_open_complaints']
            .astype(str)
            .str.split('/')
            .str[1]
            .astype(float)
        )
    
    # 3. Drop rows missing essential demographics
    df = df.dropna(subset=['income', 'education', 'state']).copy()
    
    # 4. Normalize gender
    if 'gender' in df.columns:
        df['gender'] = df['gender'].str.upper().replace({'F': 'FEMALE', 'M': 'MALE'})
    
    return df

df1_clean = clean_data(df1)
df2_clean = clean_data(df2)
df3_clean = clean_data(df3)



In [16]:
#combine them all 
dfc = pd.concat([df1_clean, df2_clean, df3_clean], ignore_index=True)

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [ ]:
# Your code goes here

dfe=pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv")

#total claim amount pivot: 
pivot_revenue = pd.pivot_table(dfe, values='total_claim_amount',index='sales_channel', aggfunc='sum').round(2)

# Sort from highest to lowest revenue
pivot_revenue = pivot_revenue.sort_values(by='total_claim_amount', ascending=False)

pivot_revenue

#insights: biggest sales channel is AGENT  WITH A TOTAL claim of 1,810,226. The channel with the least amount of sales is the web sales with 706600. 


,total_claim_amount
sales_channel,
Agent,1810226.82
Branch,1301204.00
Call Center,926600.82
Web,706600.04


In [ ]:
pivot_clv = pd.pivot_table(dfe,values='customer_lifetime_value',index=['gender', 'education'], aggfunc='mean').round(2)

pivot_clv.sort_values(by="customer_lifetime_value", ascending=False)

#Conclusions: 
 #1. The most profitable demographic group in terms of lifetime value, are Females with High School or Below with a value of 8675, followed by Males with Master with 8168 and Females with Master with 8157. 
 #2. The least profitable group are Females with Doctorate with only 7328 

customer_lifetime_value
gender education                                    
F      High School or Below                  8675.22
M      Master                                8168.83
F      Master                                8157.05
M      High School or Below                  8149.69
       College                               8052.46
F      Bachelor                              7874.27
       College                               7748.82
M      Bachelor                              7703.60
       Doctor                                7415.33
F      Doctor                                7328.51

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [47]:
# Your code goes here

dfe['month'] = pd.to_datetime(dfe['effective_to_date']).dt.month_name()

In [ ]:
complaints_summary = (dfe.groupby(['policy_type', 'month'])['number_of_open_complaints'].count().reset_index(name='complaints_count')
)
complaints_summary.head()

,policy_type,month,complaints_count
0,Corporate Auto,February,1089
1,Corporate Auto,January,1252
2,Personal Auto,February,3799
3,Personal Auto,January,4329
4,Special Auto,February,204


In [ ]:
complaints_summary = (dfe.groupby(['policy_type', 'month'])['number_of_open_complaints'].count().reset_index(name='complaints_count'))
complaints_summary.head()

#The highest complaints by policy type and month is personal auto with 4329 complaints in january and 3799 in february, the least was special auto with 204 in february

,policy_type,month,complaints_count
0,Corporate Auto,February,1089
1,Corporate Auto,January,1252
2,Personal Auto,February,3799
3,Personal Auto,January,4329
4,Special Auto,February,204
